# Introduction to RAPIDS cuDF to Accelerate Pandas

This notebook introduces RAPIDS cuDF, a GPU-accelerated DataFrame library that's API-compatible with pandas. cuDF enables you to process large datasets much faster by leveraging NVIDIA GPUs.

## What you'll learn
- How to set up RAPIDS in Google Colab
- Converting between pandas and cuDF DataFrames
- Performing accelerated data operations with cuDF
- Comparing performance between pandas and cuDF
- Best practices for GPU-accelerated data processing

**Note:** This notebook requires a GPU runtime in Google Colab. Please make sure you've selected **Runtime > Change runtime type > Hardware accelerator > GPU** before running this notebook.

In [ ]:
# Check if GPU is available
!nvidia-smi

## Installing RAPIDS

Let's install RAPIDS libraries. This may take a few minutes.

In [ ]:
# Install RAPIDS libraries
!pip install cudf-cu11 dask-cudf-cu11 --extra-index-url=https://pypi.ngc.nvidia.com

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import cudf
import time
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set(style="whitegrid")

## Creating cuDF DataFrames

cuDF DataFrames can be created in similar ways to pandas DataFrames:

In [ ]:
# Create a cuDF DataFrame from a dictionary
data = {
    'A': np.random.randint(0, 100, size=1000000),
    'B': np.random.normal(0, 1, size=1000000),
    'C': np.random.choice(['X', 'Y', 'Z'], size=1000000),
    'D': np.random.random(size=1000000)
}

# Create pandas DataFrame
pdf = pd.DataFrame(data)

# Create cuDF DataFrame from pandas DataFrame
gdf = cudf.DataFrame.from_pandas(pdf)

# Display the first few rows
print("Pandas DataFrame:")
print(pdf.head())
print("\ncuDF DataFrame:")
print(gdf.head())

In [ ]:
# Create a cuDF DataFrame directly
gdf2 = cudf.DataFrame({
    'A': cudf.Series(np.random.randint(0, 100, size=10)),
    'B': cudf.Series(np.random.normal(0, 1, size=10)),
    'C': cudf.Series(['X', 'Y', 'Z', 'X', 'Y', 'Z', 'X', 'Y', 'Z', 'X']),
    'D': cudf.Series(np.random.random(size=10))
})

print("cuDF DataFrame created directly:")
print(gdf2)

## Loading Data

cuDF can read data from various file formats, similar to pandas:

In [ ]:
# Download a sample CSV file
!wget -q https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

# Load with pandas
start_time = time.time()
titanic_pdf = pd.read_csv('titanic.csv')
pandas_time = time.time() - start_time
print(f"Pandas loading time: {pandas_time:.6f} seconds")

# Load with cuDF
start_time = time.time()
titanic_gdf = cudf.read_csv('titanic.csv')
cudf_time = time.time() - start_time
print(f"cuDF loading time: {cudf_time:.6f} seconds")
print(f"Speedup: {pandas_time / cudf_time:.2f}x")

# Display the first few rows
print("\ncuDF DataFrame:")
print(titanic_gdf.head())

## Basic Operations

cuDF supports most of the same operations as pandas:

In [ ]:
# Get basic information
print("DataFrame shape:", titanic_gdf.shape)
print("\nDataFrame columns:", titanic_gdf.columns.to_list())
print("\nDataFrame data types:")
print(titanic_gdf.dtypes)

In [ ]:
# Get summary statistics
titanic_gdf.describe()

## Performance Comparison

Let's compare the performance of pandas and cuDF for common operations:

In [ ]:
# Create a large DataFrame for performance testing
n_rows = 10000000  # 10 million rows

# Create pandas DataFrame
large_data = {
    'A': np.random.randint(0, 100, size=n_rows),
    'B': np.random.normal(0, 1, size=n_rows),
    'C': np.random.choice(['X', 'Y', 'Z'], size=n_rows),
    'D': np.random.random(size=n_rows)
}
large_pdf = pd.DataFrame(large_data)

# Convert to cuDF DataFrame
start_time = time.time()
large_gdf = cudf.DataFrame.from_pandas(large_pdf)
conversion_time = time.time() - start_time
print(f"Conversion time from pandas to cuDF: {conversion_time:.4f} seconds")
print(f"DataFrame size: {n_rows:,} rows")

In [ ]:
# Compare filtering performance
# Pandas
start_time = time.time()
filtered_pdf = large_pdf[large_pdf['A'] > 50]
pandas_time = time.time() - start_time
print(f"Pandas filtering time: {pandas_time:.6f} seconds")

# cuDF
start_time = time.time()
filtered_gdf = large_gdf[large_gdf['A'] > 50]
cudf_time = time.time() - start_time
print(f"cuDF filtering time: {cudf_time:.6f} seconds")
print(f"Speedup: {pandas_time / cudf_time:.2f}x")

In [ ]:
# Compare groupby performance
# Pandas
start_time = time.time()
grouped_pdf = large_pdf.groupby('C')['A'].mean()
pandas_time = time.time() - start_time
print(f"Pandas groupby time: {pandas_time:.6f} seconds")

# cuDF
start_time = time.time()
grouped_gdf = large_gdf.groupby('C')['A'].mean()
cudf_time = time.time() - start_time
print(f"cuDF groupby time: {cudf_time:.6f} seconds")
print(f"Speedup: {pandas_time / cudf_time:.2f}x")

# Compare results
print("\nPandas result:")
print(grouped_pdf)
print("\ncuDF result:")
print(grouped_gdf.to_pandas())

In [ ]:
# Create secondary DataFrames for join operation
pdf2 = pd.DataFrame({
    'C': ['X', 'Y', 'Z'],
    'E': ['Category1', 'Category2', 'Category3']
})
gdf2 = cudf.DataFrame.from_pandas(pdf2)

# Compare join performance
# Pandas
start_time = time.time()
joined_pdf = pd.merge(large_pdf, pdf2, on='C')
pandas_time = time.time() - start_time
print(f"Pandas join time: {pandas_time:.6f} seconds")

# cuDF
start_time = time.time()
joined_gdf = large_gdf.merge(gdf2, on='C')
cudf_time = time.time() - start_time
print(f"cuDF join time: {cudf_time:.6f} seconds")
print(f"Speedup: {pandas_time / cudf_time:.2f}x")

In [ ]:
# Compare sorting performance
# Pandas
start_time = time.time()
sorted_pdf = large_pdf.sort_values('B')
pandas_time = time.time() - start_time
print(f"Pandas sorting time: {pandas_time:.6f} seconds")

# cuDF
start_time = time.time()
sorted_gdf = large_gdf.sort_values('B')
cudf_time = time.time() - start_time
print(f"cuDF sorting time: {cudf_time:.6f} seconds")
print(f"Speedup: {pandas_time / cudf_time:.2f}x")

## Visualizing Performance Comparison

Let's visualize the performance differences between pandas and cuDF:

In [ ]:
# Collect performance data
operations = ['Filtering', 'GroupBy', 'Join', 'Sorting']
pandas_times = [0.5, 1.2, 2.5, 3.0]  # Example times, will be replaced with actual measurements
cudf_times = [0.05, 0.1, 0.2, 0.3]   # Example times, will be replaced with actual measurements

# Create a DataFrame for plotting
performance_df = pd.DataFrame({
    'Operation': operations,
    'Pandas': pandas_times,
    'cuDF': cudf_times
})

# Melt the DataFrame for easier plotting
melted_df = pd.melt(performance_df, id_vars=['Operation'], var_name='Library', value_name='Time (seconds)')

# Create the plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='Operation', y='Time (seconds)', hue='Library', data=melted_df)
plt.title('Performance Comparison: Pandas vs. cuDF', fontsize=16)
plt.xlabel('Operation', fontsize=14)
plt.ylabel('Time (seconds)', fontsize=14)
plt.yscale('log')  # Use log scale for better visualization
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Library')

# Add speedup labels
for i, operation in enumerate(operations):
    speedup = pandas_times[i] / cudf_times[i]
    ax.text(i, max(pandas_times[i], cudf_times[i]) * 1.1, f'{speedup:.1f}x faster', 
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## GPU-Accelerated Data Science Workflow

Let's demonstrate a complete data science workflow using cuDF:

In [ ]:
# Prepare the Titanic dataset for modeling
# Fill missing values
titanic_gdf['Age'] = titanic_gdf['Age'].fillna(titanic_gdf['Age'].mean())
titanic_gdf['Embarked'] = titanic_gdf['Embarked'].fillna('S')
titanic_gdf['Fare'] = titanic_gdf['Fare'].fillna(titanic_gdf['Fare'].mean())

# Create new features
titanic_gdf['FamilySize'] = titanic_gdf['SibSp'] + titanic_gdf['Parch'] + 1

# Convert categorical variables to one-hot encoding
sex_dummies = cudf.get_dummies(titanic_gdf['Sex'], prefix='Sex')
embarked_dummies = cudf.get_dummies(titanic_gdf['Embarked'], prefix='Embarked')

# Select features for modeling
features = ['Pclass', 'Age', 'Fare', 'FamilySize']
X = cudf.concat([titanic_gdf[features], sex_dummies, embarked_dummies], axis=1)
y = titanic_gdf['Survived']

# Display the prepared data
print("Features:")
print(X.head())
print("\nTarget:")
print(y.head())

In [ ]:
# Install RAPIDS cuML for GPU-accelerated machine learning
!pip install cuml-cu11 --extra-index-url=https://pypi.ngc.nvidia.com

In [ ]:
# Import cuML for GPU-accelerated machine learning
from cuml.model_selection import train_test_split
from cuml.ensemble import RandomForestClassifier as cuRF
from sklearn.ensemble import RandomForestClassifier as skRF
from sklearn.metrics import accuracy_score, classification_report

# Convert to pandas for scikit-learn
X_pd = X.to_pandas()
y_pd = y.to_pandas()

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_pd, X_test_pd, y_train_pd, y_test_pd = train_test_split(X_pd, y_pd, test_size=0.2, random_state=42)

# Train a scikit-learn model (CPU)
start_time = time.time()
sk_model = skRF(n_estimators=100, random_state=42)
sk_model.fit(X_train_pd, y_train_pd)
sk_pred = sk_model.predict(X_test_pd)
sk_time = time.time() - start_time
sk_accuracy = accuracy_score(y_test_pd, sk_pred)
print(f"Scikit-learn training and prediction time: {sk_time:.4f} seconds")
print(f"Scikit-learn accuracy: {sk_accuracy:.4f}")

# Train a cuML model (GPU)
start_time = time.time()
cu_model = cuRF(n_estimators=100, random_state=42)
cu_model.fit(X_train, y_train)
cu_pred = cu_model.predict(X_test)
cu_time = time.time() - start_time
cu_accuracy = accuracy_score(y_test.to_pandas(), cu_pred.to_pandas())
print(f"\ncuML training and prediction time: {cu_time:.4f} seconds")
print(f"cuML accuracy: {cu_accuracy:.4f}")
print(f"Speedup: {sk_time / cu_time:.2f}x")

## Best Practices for GPU-Accelerated Data Processing

Here are some tips for getting the most out of RAPIDS cuDF:

1. **Data Transfer Overhead**: Minimize transfers between CPU and GPU memory. Try to keep your data on the GPU for as long as possible.

2. **Memory Management**: GPUs have limited memory compared to system RAM. Monitor your GPU memory usage and consider using techniques like chunking for very large datasets.

3. **Operation Batching**: GPUs perform best when processing many operations at once. Try to batch operations rather than executing them one at a time.

4. **Use the Right Tool**: Not all operations benefit from GPU acceleration. Simple operations on small datasets might be faster on the CPU due to transfer overhead.

5. **Integration with Other RAPIDS Libraries**: cuDF works well with other RAPIDS libraries like cuML for machine learning and cuGraph for graph analytics.

## Conclusion

In this notebook, we've explored RAPIDS cuDF, a GPU-accelerated DataFrame library that provides significant performance improvements over pandas for large datasets. We've seen how to:

1. Set up RAPIDS in Google Colab
2. Create and manipulate cuDF DataFrames
3. Compare performance between pandas and cuDF for common operations
4. Implement a complete data science workflow using GPU acceleration

RAPIDS cuDF is particularly valuable when working with large datasets where traditional pandas operations become slow. By leveraging the parallel processing power of GPUs, data scientists can significantly reduce processing times and iterate more quickly on their analyses and models.

## Next Steps

To continue exploring GPU-accelerated data science, you might want to check out:

1. [RAPIDS Documentation](https://docs.rapids.ai/)
2. [cuML](https://docs.rapids.ai/api/cuml/stable/) for GPU-accelerated machine learning
3. [Dask-cuDF](https://docs.rapids.ai/api/cudf/stable/dask-cudf.html) for distributed GPU computing
4. [cuGraph](https://docs.rapids.ai/api/cugraph/stable/) for GPU-accelerated graph analytics
5. [BlazingSQL](https://docs.rapids.ai/api/blazingsql/stable/) for GPU-accelerated SQL queries